# 06 · Dashboard Visualization & Operational Outputs

**Step 6** — assemble the interactive intelligence outputs: Folium flood-risk map, hotspot layer, KPI gauge and the operational report artefacts consumed by the Streamlit app.

Run `streamlit run streamlit_app/app.py` for the full live dashboard.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
from src import risk_scoring, visualization as viz, utils
from src.feature_engineering import INDICATOR_NAMES
from src.utils import PROCESSED_DIR, MAPS_DIR, FIGURES_DIR

indicators = {n: np.load(PROCESSED_DIR / f'indicator_{n}.npy') for n in INDICATOR_NAMES}
surface = np.load(PROCESSED_DIR / 'model_surface.npy') if (PROCESSED_DIR / 'model_surface.npy').exists() else None
scorer = risk_scoring.build_scorer_from_artifacts(indicators, model_surface=surface, model_blend=0.35)
result = scorer.compute(rainfall_mm=utils.RAINFALL_REFERENCE_MM)

In [ ]:
viz.kpi_gauge(result.mean_frs).show()

## Interactive Folium flood-risk map

In [ ]:
m = viz.folium_risk_map(result.frs_surface, scorer.lons, scorer.lats,
                        hotspots=result.hotspots, zones=result.zone_table,
                        save_path=MAPS_DIR / 'flood_risk_map.html')
m  # renders inline in Jupyter/Zerve

In [ ]:
# Static FRS map + persisted figures
layer_hs = PROCESSED_DIR / 'layer_hillshade.npy'
hillshade = np.load(layer_hs) if layer_hs.exists() else None
viz.plot_frs(result.frs_surface, hillshade=hillshade)
print('Saved operational maps to', MAPS_DIR, 'and figures to', FIGURES_DIR)

## Launch the deployable dashboard

```bash
streamlit run streamlit_app/app.py
```

The dashboard exposes rainfall sliders, zone selection, live-monitoring simulation, dynamic KPIs, interactive maps and downloadable reports.